# 01 — Common Criteria RAG Corpus & Embedding Builder

Chunks Common Criteria Protection Profile documents (`cc_requirements_dataset.json`), extracts SFR IDs via regex, encodes chunks with `all-MiniLM-L6-v2`, saves `chunks.json` + `chunk_embeddings.npy`, and verifies semantic retrieval.

In [ ]:
!pip install -q sentence-transformers numpy

In [ ]:
import json
import re
import numpy as np
from sentence_transformers import SentenceTransformer

with open("cc_requirements_dataset.json", "r") as f:
    dataset = json.load(f)
print(f"Loaded {len(dataset)} documents.")

SFR_PATTERN = re.compile(r"\b([A-Z]{3}_[A-Z]{2,5}(?:_[A-Z]{2,5})?\.\d+(?:\.\d+)?)\b")

def chunk_document(doc, chunk_size=800, overlap=150):
    text = doc["text"]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        sfr_ids = SFR_PATTERN.findall(chunk_text)
        if len(chunk_text.strip()) > 100:
            chunks.append({
                "doc_id": doc["id"], "title": doc["title"], "category": doc["category"],
                "source_url": doc["source_url"], "text": chunk_text.strip(),
                "sfr_ids": list(set(sfr_ids)),
            })
        start += chunk_size - overlap
    return chunks

TOP_CATEGORIES = [
    "Network and Network-Related Devices and Systems",
    "Data Protection",
    "Boundary Protection Devices and Systems",
    "Trusted Computing",
    "Access Control Devices and Systems",
]

chunks = []
for doc in dataset:
    if doc["category"] in TOP_CATEGORIES:
        chunks.extend(chunk_document(doc))
print(f"Built {len(chunks)} chunks.")

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([c["text"] for c in chunks], show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings)

np.save("chunk_embeddings.npy", embeddings)
with open("chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)
print("Saved chunk_embeddings.npy and chunks.json")

def retrieve(query, top_k=3):
    q_emb = model.encode([query])[0]
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(q_emb)
    sims = embeddings @ q_emb / (norms + 1e-8)
    idx = np.argsort(-sims)[:top_k]
    return [{**chunks[i], "similarity": float(sims[i])} for i in idx]

results = retrieve("Firewall traffic filtering and access control policy enforcement")
for r in results:
    print(f"[{r['similarity']:.3f}] {r['title']} (SFRs: {r['sfr_ids']})")
    print(f"    {r['text'][:150]}...\n")